# 나이브베이즈 분류기
확률기반 분류 알고리즘

피쳐들이 서로 독립적이라고 가정.(naive)

베이즈 정리를 이용하여 어떤 클래스에 속할 확률이 가장 높은지 계산.

텍스트 분류 문제에서 효율적으로 사용.

In [1]:
# 데이터 다운로드
from sklearn.datasets import fetch_20newsgroups
newsdata = fetch_20newsgroups(subset='train')
print(newsdata.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


In [2]:
# 각 데이터의 개수를 확인
print(
    len(newsdata.data),        # 뉴스 기사 데이터 개수
    len(newsdata.filenames),   # 뉴스 파일 경로 개수
    len(newsdata.target_names),# 뉴스 카테고리 종류 개수
    len(newsdata.target)       # 각 뉴스의 카테고리 라벨 개수
)

11314 11314 20 11314


In [3]:
# 뉴스 데이터의 카테고리 이름 목록 출력
print(newsdata.target_names)   # 뉴스 카테고리 이름

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [4]:
# 첫 번째 뉴스 데이터의 카테고리 번호 출력
print(newsdata.target[0])  # 카테고리에 숫자 라벨

7


In [ ]:
# 7번 인덱스에 해당하는 뉴스 카테고리 이름 출력
print(newsdata.target_names[7])  

rec.autos


In [6]:
# 첫 번째 뉴스 기사 내용 출력
print(newsdata.data[0]) 

From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----







In [7]:
# 텍스트 데이터를 숫자 벡터로 변환하는 도구
from sklearn.feature_extraction.text import CountVectorizer

# 단어 중요도를 계산하는 TF-IDF 변환기
from sklearn.feature_extraction.text import TfidfTransformer

# 다항 분포 기반 나이브 베이즈 분류 모델
from sklearn.naive_bayes import MultinomialNB

# 모델 예측 결과의 정확도 계산
from sklearn.metrics import accuracy_score

In [ ]:
# 텍스트 데이터를 단어 빈도 기반 벡터로 변환하는 객체 생성
# 문서마다 등장하는 단어의 빈도를 기반으로 숫자 벡터 형로 변환
dtmvector = CountVectorizer()

# 뉴스 기사 텍스트를 학습하여 문서-단어 행렬(DTM) 생성
# DTM 을 TF- IDF 행렬 로 변환 : 문서 내에서 많이 등장하고 전체문서에서는 
# 드물게 등장하는 단어에 더 높은은 가중치를 부여

X_train_dtm = dtmvector.fit_transform(newsdata.data)

# 생성된 문서-단어 행렬의 크기 출력 (문서 수, 단어 수)
print(X_train_dtm.shape)

(11314, 130107)


In [9]:
# TF-IDF 변환기 객체 생성
tfidf_transformer = TfidfTransformer()

# 단어 빈도 행렬(DTM)을 TF-IDF 값으로 변환
tfidfv = tfidf_transformer.fit_transform(X_train_dtm)

# 변환된 TF-IDF 행렬의 크기 출력 (문서 수, 단어 수)
print(tfidfv.shape)

(11314, 130107)


In [ ]:
# 나이브 베이즈 모델 학습 
# 단어 등장 확률 기반으로 어떤 카테고리에 속하는지 예측
# 다항 분포 기반 나이브 베이즈 모델 생성
mod = MultinomialNB()

# TF-IDF 데이터와 정답 라벨을 사용해 모델 학습
mod.fit(tfidfv, newsdata.target)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [11]:
MultinomialNB(
    alpha=1.0,        # 라플라스 스무딩 값 (0이 되지 않도록 보정)
    class_prior=None, # 클래스의 사전 확률 (None이면 자동 계산)
    fit_prior=True    # 데이터로부터 클래스 확률을 학습할지 여부
)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [ ]:
# 테스트용 뉴스 데이터 불러오기
newsdata_test = fetch_20newsgroups(subset='test', shuffle=True) 

# 테스트 데이터를 문서-단어 행렬(DTM)로 변환
X_test_dtm = dtmvector.transform(newsdata_test.data)  

# DTM을 TF-IDF 행렬로 변환
tfidfv_test = tfidf_transformer.transform(X_test_dtm)  

# 테스트 데이터에 대한 예측
predicted = mod.predict(tfidfv_test)  

# 정확도 계산
print("정확도 :", accuracy_score(newsdata_test.target, predicted))  

정확도 : 0.7738980350504514
